In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

مرحلة بناء مدير الحالة واختباره

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive-Tutor-RL/src/state_manager/session_tracker.py
import pandas as pd
import random

class SessionTracker:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.current_level = 'A1'
        self.consecutive_knows = 0
        self.history = []

    def get_word(self):
        # تصفية الكلمات بناءً على المستوى الحالي
        level_words = self.df[self.df['level'] == self.current_level]
        if level_words.empty:
            return None
        return random.choice(level_words['word'].values)

    def process_response(self, word, knows):
        if knows:
            self.consecutive_knows += 1
            if self.consecutive_knows >= 3:
                self.upgrade_level()
        else:
            self.consecutive_knows = 0
            return "TRIGGER_NEURAL_MODEL"
        return "CONTINUE"

    def upgrade_level(self):
        levels = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
        current_idx = levels.index(self.current_level)
        if current_idx < len(levels) - 1:
            self.current_level = levels[current_idx + 1]
            self.consecutive_knows = 0
            print(f"🎉 المستوى تمت ترقيته إلى: {self.current_level}")

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Adaptive_Tutor_RL/src')
from state_manager.session_tracker import SessionTracker

# تهيئة مدير الحالة
tracker = SessionTracker('/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/repo_words.csv')

print(f"بدء الجلسة في المستوى: {tracker.current_level}")

# محاكاة: الطالب يعرف 3 كلمات (ليتم ترقيته)
for i in range(3):
    word = tracker.get_word()
    print(f"السؤال: هل تعرف معنى كلمة '{word}'؟ (نعم/لا)")
    # سنفترض هنا أن الطالب أجاب بـ "نعم"
    result = tracker.process_response(word, knows=True)
    print(f"الحالة: {result}")

# محاكاة: الطالب لا يعرف كلمة
word = tracker.get_word()
print(f"السؤال: هل تعرف معنى كلمة '{word}'؟")
result = tracker.process_response(word, knows=False)
print(f"الحالة: {result}") # يجب أن يطبع هنا TRIGGER_NEURAL_MODEL